# SuperStore Sales & Profitability: Where Is the Business Actually Losing Money?

**Author:** Subhranshu Panda &nbsp;|&nbsp; **Type:** Business-question EDA + time-series forecast &nbsp;|&nbsp; **Tools:** Python, Pandas, Matplotlib/Seaborn/Plotly, Statsmodels (SARIMA)

**The business problem:** most public "Superstore" notebooks stop at "which category sells the most". This one goes further and asks what a stakeholder actually needs answered: which segments/products are *profitable* (not just high-revenue), does discounting help or hurt margin, which regions look strong on sales but are quietly unprofitable, and what does the next 6 months of sales look like.

**Dataset:** ~9,800 order line items from a US retail superstore, 2015–2018.

**Headline findings:**
- **Furniture is a volume trap** — #2 category by revenue, dead last by profit; Tables (-9.0% margin) and Bookcases (-3.0% margin) are outright loss-making.
- **Discounts above ~20% destroy profit** — average profit per order goes negative past that threshold, and the 30–50% discount band loses ~$157/order on average.
- **Texas, Ohio, Pennsylvania and Illinois are profitability blind spots** — all top-10 states by sales, all net-negative on profit.
- A SARIMA forecast projects the next 6 months of sales, including the seasonal Jan/Feb dip seen every prior year.

*This notebook is self-contained and runs top-to-bottom on Kaggle with "Run All" — no external database, no local package install. For the fully engineered version of this project (installable Python package, pytest suite, CI/CD, Docker image, live Streamlit dashboard), see the [GitHub repo](https://github.com/SubhranshuPan/Super-Store-Data-Analysis-Project) linked at the end.*

---


# SuperStore Sales & Profitability Analysis

**Author:** Subhranshu Panda &nbsp;|&nbsp; **Type:** Exploratory Data Analysis (EDA) &nbsp;|&nbsp; **Tools:** Python, Pandas, Matplotlib, Seaborn, Plotly

This notebook analyses 9,800 order line items from a US retail superstore (2015-01-03 to 2018-12-30) to answer five business questions:

1. Which customer segments and individuals drive the most revenue and lifetime value?
2. Which product categories/sub-categories are profitable, and which are quietly losing money?
3. Does discounting help or hurt profit?
4. Which regions are strong on sales but weak on profit?
5. How has the business trended year over year, and what does that imply going forward?

**Note on data:** the source file only shipped with `Sales` (no `Profit`, `Discount`, or `Quantity`). Those three columns were sourced from the public reference "Sample – Superstore" dataset and merged in on the shared `Row ID` key (verified as an exact 1:1 match on Product ID, Customer and Sales value for every row) so that a real profitability analysis — not just a revenue one — is possible. See `data/README.md` for the merge methodology.


In [ ]:
import warnings

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import seaborn as sns

PALETTE = "viridis"
sns.set_theme(style="whitegrid")


# --- Inlined from superstore/data.py & superstore/forecast.py (kept identical to the
#     package versions in the GitHub repo, just copied in-line so this notebook has
#     zero local-package dependency and runs standalone on Kaggle) ---

def load_data(path):
    return pd.read_csv(path)


def clean_data(df):
    df = df.copy()
    df["Postal Code"] = df["Postal Code"].fillna(0).astype(int)
    df["Order Date"] = pd.to_datetime(df["Order Date"], dayfirst=True)
    df["Ship Date"] = pd.to_datetime(df["Ship Date"], dayfirst=True)
    df["Order to Ship Days"] = (df["Ship Date"] - df["Order Date"]).dt.days
    df = df.drop_duplicates()
    return df


def forecast_monthly_sales(df, periods=6):
    from statsmodels.tsa.statespace.sarimax import SARIMAX

    monthly = df.set_index("Order Date").resample("ME")["Sales"].sum()

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        model = SARIMAX(
            monthly, order=(1, 1, 1), seasonal_order=(1, 1, 1, 12),
            enforce_stationarity=False, enforce_invertibility=False,
        )
        fit = model.fit(disp=False)
        pred = fit.get_forecast(steps=periods)

    forecast_index = pd.date_range(monthly.index[-1] + pd.offsets.MonthEnd(1), periods=periods, freq="ME")
    forecast_mean = pd.Series(pred.predicted_mean.values, index=forecast_index)
    ci = pred.conf_int(alpha=0.2)
    lower = pd.Series(ci.iloc[:, 0].values, index=forecast_index)
    upper = pd.Series(ci.iloc[:, 1].values, index=forecast_index)

    out = pd.DataFrame({
        "Sales": monthly.reindex(monthly.index.union(forecast_index)),
        "Forecast": pd.Series(index=monthly.index, dtype=float).reindex(monthly.index.union(forecast_index)),
    })
    out.loc[monthly.index, "Forecast"] = monthly.values
    out.loc[forecast_index, "Forecast"] = forecast_mean.values
    out.loc[forecast_index, "Lower CI"] = lower.values
    out.loc[forecast_index, "Upper CI"] = upper.values
    return out


## 1. Load & Clean the Data

In [ ]:
import os
import glob

if os.path.exists("/kaggle/input"):
    _matches = glob.glob("/kaggle/input/**/superstore_sales.csv", recursive=True)
    DATA_PATH = _matches[0] if _matches else "../data/superstore_sales.csv"
else:
    DATA_PATH = "../data/superstore_sales.csv"

df = load_data(DATA_PATH)
df.head()


In [ ]:
df.info()


**Cleaning steps:**
- `Postal Code` had 11 missing values → filled with `0` and cast to `int` (these are placeholder/unknown ZIPs, not real gaps in sales data).
- `Order Date` / `Ship Date` parsed as datetimes (day-first, since the source is UK-style `dd/mm/yyyy`).
- Checked for exact duplicate rows.


In [ ]:
n_dupes = df.duplicated().sum()
print(f"Duplicate rows: {n_dupes}")

df = clean_data(df)  # fills Postal Code, parses dates, derives Order to Ship Days, dedupes
df.shape


## 2. Executive Summary

| Metric | Value |
|---|---|
| Total Sales | $2,261,537 |
| Total Profit | $278,979 |
| Overall Profit Margin | 12.3% |
| Orders | 4,922 |
| Unique Customers | 793 |
| Avg. Order Value | $459 |
| Repeat Customer Rate | 98.4% |

**Headline findings** (each is unpacked with evidence in the sections below):

- **Furniture is a volume trap.** It's the #2 category by revenue but dead last by profit — margin is barely above breakeven, and **Tables and Bookcases are outright loss-making** (-9.0% and -3.0% margin respectively).
- **Discounts above ~20% destroy profit.** Average profit per order turns negative once discount exceeds 0.2, and the 30–50% discount band loses an average of $157 per order.
- **Texas is a profitability blind spot.** It's the 3rd largest state by sales but the single least profitable state overall, alongside Ohio, Pennsylvania and Illinois.
- **Growth resumed strongly after a 2016 dip:** sales fell 4.3% in 2016 then grew 30.6% and 20.3% in the following two years.
- **Retention is high (98%+ of customers reorder)**, so the bigger lever is basket size/margin per order, not acquisition.


## 3. Customer Segmentation

In [ ]:
segment_counts = df["Segment"].value_counts()
segment_sales = df.groupby("Segment")["Sales"].sum().sort_values(ascending=False)
segment_profit = df.groupby("Segment")["Profit"].sum().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].pie(segment_counts, labels=segment_counts.index, autopct="%1.1f%%",
            colors=sns.color_palette(PALETTE, len(segment_counts)))
axes[0].set_title("Customers by Segment")
axes[1].bar(segment_sales.index, segment_sales.values, color=sns.color_palette(PALETTE, len(segment_sales)))
axes[1].set_title("Total Sales by Segment ($)")
axes[1].yaxis.set_major_formatter(mticker.StrMethodFormatter("{x:,.0f}"))
plt.tight_layout()
plt.show()

print(segment_sales)
print(segment_profit)


**Reading it:** Consumers are the largest segment by both headcount and revenue, but Corporate customers generate the highest average lifetime value per head (see below) — a smaller, higher-value segment worth protecting with account management rather than discounting.

## 4. Customer Lifetime Value (CLTV)

In [ ]:
# CLTV proxy = total historical Sales per customer (no churn/margin model available,
# so this is a simple "revenue-to-date" CLTV rather than a predictive one).
customer_revenue = df.groupby(["Segment", "Customer ID"])["Sales"].sum().reset_index()
cltv_by_segment = customer_revenue.groupby("Segment")["Sales"].mean().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(cltv_by_segment.index, cltv_by_segment.values, color=sns.color_palette(PALETTE, len(cltv_by_segment)))
ax.set_title("Average Customer Lifetime Value by Segment")
ax.set_ylabel("Avg total Sales per customer ($)")
ax.yaxis.set_major_formatter(mticker.StrMethodFormatter("{x:,.0f}"))
plt.tight_layout()
plt.show()

cltv_by_segment.round(2)


In [ ]:
top_spenders = (
    df.groupby(["Customer ID", "Customer Name"])["Sales"].sum()
    .sort_values(ascending=False).head(10).reset_index()
)
top_spenders


## 5. RFM Segmentation (Recency, Frequency, Monetary)

A simple RFM model buckets every customer into a quartile (1-4) for each of Recency, Frequency and Monetary value, sums the three scores (3-12), and maps the total onto a tier. This is the standard lightweight approach for prioritising CRM/marketing spend without needing a churn model.

In [ ]:
snapshot_date = df["Order Date"].max() + pd.Timedelta(days=1)

rfm = df.groupby("Customer ID").agg(
    Recency=("Order Date", lambda x: (snapshot_date - x.max()).days),
    Frequency=("Order ID", "nunique"),
    Monetary=("Sales", "sum"),
).reset_index()

rfm["R"] = pd.qcut(rfm["Recency"], 4, labels=range(4, 0, -1)).astype(int)
rfm["F"] = pd.qcut(rfm["Frequency"].rank(method="first"), 4, labels=range(1, 5)).astype(int)
rfm["M"] = pd.qcut(rfm["Monetary"], 4, labels=range(1, 5)).astype(int)
rfm["RFM_Score"] = rfm["R"] + rfm["F"] + rfm["M"]


def tier(score):
    if score >= 10:
        return "Champions"
    if score >= 8:
        return "Loyal Customers"
    if score >= 6:
        return "Potential Loyalists"
    if score >= 4:
        return "At Risk"
    return "Lost / Dormant"


rfm["Customer Tier"] = rfm["RFM_Score"].apply(tier)
tier_order = ["Champions", "Loyal Customers", "Potential Loyalists", "At Risk", "Lost / Dormant"]
tier_counts = rfm["Customer Tier"].value_counts().reindex(tier_order)

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.barh(tier_counts.index, tier_counts.values, color=sns.color_palette(PALETTE, len(tier_counts)))
ax.set_title("Customers by RFM Tier")
ax.set_xlabel("Number of customers")
plt.tight_layout()
plt.show()

tier_counts


**Reading it:** roughly a third of the customer base are Champions or Loyal Customers — a healthy core. The `At Risk` and `Lost / Dormant` tiers are the natural target list for a win-back campaign; `rfm.to_csv` below exports the full per-customer table for that purpose.

In [ ]:
import os

REPORTS_DIR = "/kaggle/working" if os.path.exists("/kaggle/input") else "../reports"
os.makedirs(REPORTS_DIR, exist_ok=True)

rfm_path = os.path.join(REPORTS_DIR, "rfm_customer_segments.csv")
rfm.to_csv(rfm_path, index=False)
print(f"Exported {len(rfm)} customer RFM records to {rfm_path}")


## 6. Shipping Mode Analysis

In [ ]:
ship_counts = df["Ship Mode"].value_counts()
avg_ship_days = df.groupby("Ship Mode")["Order to Ship Days"].mean().round(1).sort_values()

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].pie(ship_counts, labels=ship_counts.index, autopct="%1.1f%%",
            colors=sns.color_palette(PALETTE, len(ship_counts)))
axes[0].set_title("Orders by Shipping Mode")
axes[1].bar(avg_ship_days.index, avg_ship_days.values, color=sns.color_palette(PALETTE, len(avg_ship_days)))
axes[1].set_title("Avg. Days from Order to Ship")
plt.tight_layout()
plt.show()

print(ship_counts)
print(avg_ship_days)


**Reading it:** 60% of orders ship Standard Class, averaging 5 days — the slowest option. Same Day is same-day by definition. There's an implicit customer-experience trade-off worth flagging to Ops: the cheapest mode is also the one most customers default into, whether by choice or because it's the default at checkout.

## 7. Geographic Analysis

In [ ]:
state_sales = df.groupby("State")["Sales"].sum().sort_values(ascending=False)
state_profit = df.groupby("State")["Profit"].sum().sort_values()

fig, ax = plt.subplots(figsize=(8, 6))
top15 = state_sales.head(15).sort_values()
ax.barh(top15.index, top15.values, color=sns.color_palette(PALETTE, len(top15)))
ax.set_title("Top 15 States by Total Sales")
ax.xaxis.set_major_formatter(mticker.StrMethodFormatter("{x:,.0f}"))
plt.tight_layout()
plt.show()

print("Top 5 states by sales:\n", state_sales.head())
print("\nLeast profitable states:\n", state_profit.head())


In [ ]:
all_state_mapping = {
    "Alabama": "AL", "Alaska": "AK", "Arizona": "AZ", "Arkansas": "AR",
    "California": "CA", "Colorado": "CO", "Connecticut": "CT", "Delaware": "DE",
    "Florida": "FL", "Georgia": "GA", "Hawaii": "HI", "Idaho": "ID", "Illinois": "IL",
    "Indiana": "IN", "Iowa": "IA", "Kansas": "KS", "Kentucky": "KY", "Louisiana": "LA",
    "Maine": "ME", "Maryland": "MD", "Massachusetts": "MA", "Michigan": "MI", "Minnesota": "MN",
    "Mississippi": "MS", "Missouri": "MO", "Montana": "MT", "Nebraska": "NE", "Nevada": "NV",
    "New Hampshire": "NH", "New Jersey": "NJ", "New Mexico": "NM", "New York": "NY",
    "North Carolina": "NC", "North Dakota": "ND", "Ohio": "OH", "Oklahoma": "OK",
    "Oregon": "OR", "Pennsylvania": "PA", "Rhode Island": "RI", "South Carolina": "SC",
    "South Dakota": "SD", "Tennessee": "TN", "Texas": "TX", "Utah": "UT", "Vermont": "VT",
    "Virginia": "VA", "Washington": "WA", "West Virginia": "WV", "Wisconsin": "WI", "Wyoming": "WY",
}

profit_by_state = df.groupby("State")["Profit"].sum().reset_index()
profit_by_state["Abbreviation"] = profit_by_state["State"].map(all_state_mapping)

fig = go.Figure(data=go.Choropleth(
    locations=profit_by_state["Abbreviation"],
    locationmode="USA-states",
    z=profit_by_state["Profit"],
    colorscale="RdYlGn",
    zmid=0,
    hoverinfo="location+z",
    showscale=True,
))
fig.update_geos(projection_type="albers usa")
fig.update_layout(geo_scope="usa", title_text="Total Profit by U.S. State (red = loss-making)")
fig.show()


**Reading it:** California and New York dominate on raw sales, which is unsurprising given population and customer count. The more actionable finding is the profit map above — Texas, Ohio, Pennsylvania and Illinois all post **negative** total profit despite being top-10 states by revenue, meaning the current pricing/discount policy in those states is subsidising volume at a loss.

## 8. Product Category & Profitability Analysis

In [ ]:
cat_sales = df.groupby("Category")["Sales"].sum().sort_values(ascending=False)
cat_profit = df.groupby("Category")["Profit"].sum().sort_values(ascending=False)
cat_margin = (cat_profit / cat_sales * 100).round(1)

fig, ax = plt.subplots(figsize=(6, 4))
ax.pie(cat_sales, labels=cat_sales.index, autopct="%1.1f%%", colors=sns.color_palette(PALETTE, len(cat_sales)))
ax.set_title("Sales Share by Category")
plt.tight_layout()
plt.show()

pd.DataFrame({"Sales": cat_sales, "Profit": cat_profit, "Margin %": cat_margin})


In [ ]:
subcat = df.groupby(["Category", "Sub-Category"]).agg(Sales=("Sales", "sum"), Profit=("Profit", "sum")).reset_index()
subcat["Profit Margin %"] = (subcat["Profit"] / subcat["Sales"] * 100).round(1)
subcat_sorted = subcat.sort_values("Profit")

fig, ax = plt.subplots(figsize=(8, 6))
colors = ["#d62728" if v < 0 else sns.color_palette(PALETTE, 1)[0] for v in subcat_sorted["Profit"]]
ax.barh(subcat_sorted["Sub-Category"], subcat_sorted["Profit"], color=colors)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_title("Profit by Sub-Category (red = loss-making)")
ax.xaxis.set_major_formatter(mticker.StrMethodFormatter("{x:,.0f}"))
plt.tight_layout()
plt.show()

subcat_sorted[subcat_sorted["Profit"] < 0]


In [ ]:
df_summary = df.groupby(["Category", "Sub-Category"])["Sales"].sum().reset_index()
fig = px.sunburst(
    df_summary, path=["Category", "Sub-Category"], values="Sales",
    title="Sales Breakdown: Category > Sub-Category",
)
fig.show()


**Reading it:** Technology carries the business — highest sales *and* highest profit. Furniture is the problem child: **Tables and Bookcases are structurally unprofitable**, most likely driven by high shipping/handling cost relative to price and (as the next section shows) heavier discounting on big-ticket furniture items.

## 9. Discount Impact on Profit

In [ ]:
discount_bins = pd.cut(df["Discount"], bins=[-0.01, 0, 0.1, 0.2, 0.3, 0.5, 1.0])
avg_profit_by_discount = df.groupby(discount_bins, observed=True)["Profit"].mean()

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.bar([str(i) for i in avg_profit_by_discount.index], avg_profit_by_discount.values,
       color=sns.color_palette(PALETTE, len(avg_profit_by_discount)))
ax.axhline(0, color="black", linewidth=0.8)
ax.set_title("Average Profit per Order by Discount Band")
ax.set_xlabel("Discount band")
ax.set_ylabel("Avg profit ($)")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

avg_profit_by_discount.round(2)


**Reading it:** average profit per order is positive with no discount or a light (0-10%) discount, turns marginal at 10-20%, and goes **negative** beyond 20% — the 30-50% band loses money on average. This is a direct, actionable finding: discounting policy above ~20% should be reviewed, particularly since it concentrates in Furniture (see above).

## 10. Time Series Analysis

In [ ]:
yearly_sales = df.groupby(df["Order Date"].dt.year)["Sales"].sum()
yoy_growth = (yearly_sales.pct_change() * 100).round(1)

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(yearly_sales.index, yearly_sales.values, marker="o", linewidth=2, color=sns.color_palette(PALETTE, 1)[0])
ax.set_title("Total Sales by Year")
ax.yaxis.set_major_formatter(mticker.StrMethodFormatter("{x:,.0f}"))
ax.set_xticks(list(yearly_sales.index))
plt.tight_layout()
plt.show()

pd.DataFrame({"Total Sales": yearly_sales, "YoY Growth %": yoy_growth})


In [ ]:
monthly_sales = df.set_index("Order Date").resample("ME")["Sales"].sum()

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(monthly_sales.index, monthly_sales.values, marker="o", markersize=3, linewidth=1.5,
        color=sns.color_palette(PALETTE, 1)[0])
ax.set_title("Monthly Sales Trend (2015-2018)")
ax.yaxis.set_major_formatter(mticker.StrMethodFormatter("{x:,.0f}"))
plt.tight_layout()
plt.show()


In [ ]:
monthly_seasonality = df.groupby(df["Order Date"].dt.month)["Sales"].mean()

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(monthly_seasonality.index, monthly_seasonality.values, color=sns.color_palette(PALETTE, 12))
ax.set_title("Average Sales by Calendar Month (seasonality)")
ax.set_xlabel("Month")
ax.set_xticks(range(1, 13))
plt.tight_layout()
plt.show()


**Reading it:** sales dipped -4.3% in 2016 before rebounding +30.6% in 2017 and +20.3% in 2018 — two consecutive years of double-digit growth. Seasonality is clear too: Q4 (Sep-Dec) consistently outperforms the rest of the year, consistent with the typical B2B/B2C year-end purchasing cycle.

In [ ]:
df_summary = df.groupby(["Category", "Ship Mode", "Sub-Category"])["Sales"].sum().reset_index()
fig = px.treemap(df_summary, path=["Category", "Ship Mode", "Sub-Category"], values="Sales",
                  title="Sales by Category > Ship Mode > Sub-Category")
fig.show()


## 11. Sales Forecast

A seasonal ARIMA (SARIMAX) model, fit on the full monthly sales history, gives a lightweight forward-looking view without needing a heavier forecasting library. This is meant as a directional planning input (e.g. for inventory/staffing), not a precise revenue prediction.

In [ ]:
forecast_df = forecast_monthly_sales(df, periods=6)

hist = forecast_df.dropna(subset=["Sales"])
fut = forecast_df[forecast_df["Sales"].isna()]

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(hist.index, hist["Sales"], marker="o", markersize=3, linewidth=1.5,
        color=sns.color_palette(PALETTE, 1)[0], label="Actual")
ax.plot(forecast_df.index, forecast_df["Forecast"], linestyle="--", linewidth=1.5,
        color="#d62728", label="Forecast")
ax.fill_between(fut.index, fut["Lower CI"], fut["Upper CI"], color="#d62728", alpha=0.15, label="80% CI")
ax.set_title("Monthly Sales: Actual + 6-Month SARIMA Forecast")
ax.yaxis.set_major_formatter(mticker.StrMethodFormatter("{x:,.0f}"))
ax.legend()
plt.tight_layout()
plt.show()

fut[["Forecast", "Lower CI", "Upper CI"]].round(0)


**Reading it:** the model projects a seasonal dip in the first two months of 2019 (consistent with the January/February lull visible every year in the historical data), before recovering. The 80% confidence interval widens quickly, as expected with only 4 years of monthly history — treat this as a planning signal, not a committed number.

## 12. Conclusion & Recommendations

**What's working:** Technology and Office Supplies are healthy, profitable categories. Retention is excellent (98%+ repeat rate) and the business has grown for two straight years after the 2016 dip.

**What needs attention:**

1. **Cap discounts on Furniture, especially Tables and Bookcases**, at or below ~20% — beyond that threshold the average order is unprofitable. This single change targets the clearest loss driver identified in this analysis.
2. **Review pricing/cost-to-serve in Texas, Ohio, Pennsylvania and Illinois** — all four are top-10 states by sales but net-negative on profit, suggesting the issue is regional pricing/discounting rather than demand.
3. **Prioritise the "At Risk" and "Lost / Dormant" RFM segments** (~200 customers) for a targeted win-back campaign — the segment-level CLTV numbers show Corporate customers are worth protecting first.
4. **Plan inventory/staffing for the Q4 seasonal peak**, which consistently accounts for a disproportionate share of annual sales.

This notebook, the reusable analysis script (`scripts/run_analysis.py`), and an interactive dashboard (`dashboard/app.py`) together form the full project — see the repo `README.md` for how to run each.


---

## Full Engineering Version, Live Dashboard & Contact

This notebook is the analysis layer of a larger, production-style project:

- **Full repo** (installable `superstore` package, 9-test pytest suite, GitHub Actions CI/CD, Dockerfile, versioned releases): [github.com/SubhranshuPan/Super-Store-Data-Analysis-Project](https://github.com/SubhranshuPan/Super-Store-Data-Analysis-Project)
- **Interactive Streamlit dashboard** (filter by date/segment/category/region, live KPIs, forecast tab): see the "Live dashboard" link in the repo README
- **One-page executive summary:** [`reports/Executive_Summary.pdf`](https://github.com/SubhranshuPan/Super-Store-Data-Analysis-Project/blob/main/reports/Executive_Summary.pdf) in the repo
- **Contact:** [subhranshu599@gmail.com](mailto:subhranshu599@gmail.com) &nbsp;|&nbsp; open to Data Analyst / Data Science part-time and internship roles (UK)
